In [1]:
# Cell 1: Imports & config

import os
import numpy as np
import pandas as pd

import torch
import torch.backends.cudnn as cudnn

# Disable cuDNN RNNs – avoids the "RNN backward can only be called in training mode" bug
cudnn.enabled = False

import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer as TFTModel
from pytorch_forecasting.metrics import MAE

print("Torch version:", torch.__version__)
print("Lightning version:", pl.__version__)

DATA_PATH = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"


Torch version: 2.5.1+cu121
Lightning version: 2.5.6


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
# Cell 2: Load basic OHLC data and create mid_close, time features

df = pd.read_parquet(DATA_PATH)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

# Ensure time is datetime
df["time"] = pd.to_datetime(df["time"])

# Sort
df = df.sort_values("time").reset_index(drop=True)

# Create a consistent close price for modeling
df["close"] = df["mid_c"]   # you can also choose bid_c or ask_c later

# Add basic temporal features
df["hour"] = df["time"].dt.hour.astype(str)
df["day_of_week"] = df["time"].dt.dayofweek.astype(str)

# Add series_id (only one pair)
df["series_id"] = "eurusd"

# Add time_idx
df["time_idx"] = np.arange(len(df), dtype=int)

df.head()


Raw shape: (61473, 14)
Columns: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,close,hour,day_of_week,series_id,time_idx
0,2016-01-07 00:00:00+00:00,542,1.07764,1.07832,1.07744,1.07778,1.07757,1.07823,1.07735,1.07770,1.07772,1.07840,1.07752,1.07787,1.07778,0,3,eurusd,0
1,2016-01-07 01:00:00+00:00,3167,1.07776,1.08100,1.07748,1.08029,1.07768,1.08092,1.07740,1.08020,1.07784,1.08109,1.07756,1.08038,1.08029,1,3,eurusd,1
2,2016-01-07 02:00:00+00:00,1567,1.08026,1.08176,1.07996,1.08152,1.08018,1.08169,1.07987,1.08144,1.08035,1.08184,1.08005,1.08159,1.08152,2,3,eurusd,2
3,2016-01-07 03:00:00+00:00,914,1.08156,1.08257,1.08150,1.08187,1.08147,1.08249,1.08142,1.08178,1.08164,1.08265,1.08157,1.08196,1.08187,3,3,eurusd,3
4,2016-01-07 04:00:00+00:00,649,1.08190,1.08256,1.08156,1.08236,1.08182,1.08247,1.08147,1.08228,1.08199,1.08264,1.08163,1.08245,1.08236,4,3,eurusd,4


In [4]:
# Cell 3: Ensure target_return exists (log-return of next hour)

if "target_return" not in df.columns:
    # 🔧 If your price column is not "close", change PRICE_COL accordingly
    PRICE_COL = "close"  # e.g. could be "midclose" or "bid"
    if PRICE_COL not in df.columns:
        raise ValueError(f"Column {PRICE_COL} not found. Please set PRICE_COL to your price column.")
    
    df["target_return"] = np.log(df[PRICE_COL].shift(-1)) - np.log(df[PRICE_COL])
    df = df.dropna(subset=["target_return"]).reset_index(drop=True)

print("Has target_return:", "target_return" in df.columns)
print(df[["time", "target_return"]].head())


Has target_return: True
                       time  target_return
0 2016-01-07 00:00:00+00:00       0.002326
1 2016-01-07 01:00:00+00:00       0.001138
2 2016-01-07 02:00:00+00:00       0.000324
3 2016-01-07 03:00:00+00:00       0.000453
4 2016-01-07 04:00:00+00:00       0.000333


In [5]:
# Cell 4: Build FEATURE_COLS_EXT automatically from numeric columns

num_cols = df.select_dtypes(include=["number", "float", "int"]).columns.tolist()

exclude_cols = [
    "time_idx",
    "target_return",
    # if your file has other numeric columns you want to exclude, add here
]

FEATURE_COLS_EXT = [c for c in num_cols if c not in exclude_cols]

print("Numeric columns:", num_cols)
print("FEATURE_COLS_EXT (used as real-valued inputs):", FEATURE_COLS_EXT)


Numeric columns: ['volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c', 'close', 'time_idx', 'target_return']
FEATURE_COLS_EXT (used as real-valued inputs): ['volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c', 'close']


In [6]:
# Cell 5: Time-based split into train / val / test

df = df.sort_values("time_idx").reset_index(drop=True)

N = len(df)
train_end = int(N * 0.7)
val_end   = int(N * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print("Total:", N)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("time_idx ranges:")
print(" train:", train_df["time_idx"].min(), "->", train_df["time_idx"].max())
print(" val  :", val_df["time_idx"].min(),   "->", val_df["time_idx"].max())
print(" test :", test_df["time_idx"].min(),  "->", test_df["time_idx"].max())


Total: 61472
Train: 43030 Val: 9221 Test: 9221
time_idx ranges:
 train: 0 -> 43029
 val  : 43030 -> 52250
 test : 52251 -> 61471


In [7]:
# Cell 6: TimeSeriesDataSet for TFT

MAX_ENCODER_LENGTH = 168   # 7 days of H1 history
MAX_PREDICTION_LENGTH = 1
BATCH_SIZE = 1024           # reduce to 128 if GPU memory is tight

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="target_return",
    group_ids=["series_id"],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    time_varying_unknown_reals=FEATURE_COLS_EXT,
    time_varying_known_categoricals=["hour", "day_of_week"],
    static_categoricals=["series_id"],

    target_normalizer=None,        # keep raw log-returns (compatible with live engine)
    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    data=val_df,
    stop_randomization=True,
)

test = TimeSeriesDataSet.from_dataset(
    training,
    data=test_df,
    stop_randomization=True,
)

len(training), len(validation), len(test)


(42862, 9053, 9053)

In [8]:
# Cell 7: Dataloaders (use TimeSeriesDataSet.to_dataloader)

train_loader = training.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0,     # Windows + Forecasting => safer with 0
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

test_loader = test.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

len(train_loader), len(val_loader), len(test_loader)


(41, 9, 9)

In [9]:
# Cell 8: Build TFT model (v3)

hidden_size = 32
attention_heads = 4
dropout = 0.10
hidden_continuous_size = 16
lstm_layers = 2
LEARNING_RATE = 1e-3

tft = TFTModel.from_dataset(
    training,
    learning_rate=LEARNING_RATE,

    hidden_size=hidden_size,
    lstm_layers=lstm_layers,
    attention_head_size=attention_heads,
    dropout=dropout,
    hidden_continuous_size=hidden_continuous_size,

    loss=MAE(),          # MAE in log-return units
    output_size=1,       # 1-step ahead regression

    log_interval=50,
    log_val_interval=1,
    reduce_on_plateau_patience=4,
)

print("TFT v3 model created.")
print("Number of parameters:", sum(p.numel() for p in tft.parameters()))


TFT v3 model created.
Number of parameters: 107600


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [10]:
# Cell 9: Trainer configuration

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="eurusd_h1_tft_v3-{epoch:02d}-{val_loss:.6f}",
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    min_delta=1e-5,
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"

trainer = pl.Trainer(
    max_epochs=80,
    accelerator=accelerator,
    devices=1,
    gradient_clip_val=0.1,
    precision=32,                     # 32-bit to avoid Half overflow
    callbacks=[checkpoint_cb, earlystop_cb],
    log_every_n_steps=20,
)

trainer


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [11]:
# Cell 10: Training

trainer.fit(
    model=tft,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print("Best checkpoint path:", checkpoint_cb.best_model_path)


You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 241    | train
3  | prescalers                         | ModuleDict                      | 512    | train
4  | static_variable_selection

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


Epoch 45: 100%|██████████| 41/41 [01:52<00:00,  0.37it/s, v_num=46, train_loss_step=0.00227, val_loss=0.000974, train_loss_epoch=0.00216]
Best checkpoint path: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_46\checkpoints\eurusd_h1_tft_v3-epoch=45-val_loss=0.000974.ckpt


In [12]:
# Cell 11: Evaluation on test set

ckpt_path = checkpoint_cb.best_model_path
print("Using checkpoint:", ckpt_path)

best_model = TFTModel.load_from_checkpoint(ckpt_path)

# Predictions
preds_raw = best_model.predict(test_loader)      # tensor [N, 1]
preds = preds_raw.detach().cpu().reshape(-1)

# True targets
all_targets = []
for batch in test_loader:
    x, y_tuple = batch          # (x, (y, weight))
    y = y_tuple[0]              # extract target tensor
    all_targets.append(y.detach().cpu().reshape(-1))

targets = torch.cat(all_targets)

preds_np = preds.numpy()
targets_np = targets.numpy()

# Metrics
mae = np.mean(np.abs(preds_np - targets_np))
print(f"Test MAE (log-return units): {mae:.8f}")

pred_dir = np.sign(preds_np)
true_dir = np.sign(targets_np)
dir_acc = (pred_dir == true_dir).mean()
print(f"Direction accuracy: {dir_acc:.4f} ({dir_acc*100:.2f}%)")

print("\nSample comparison (first 5):")
for i in range(5):
    print(f"{i}: pred={preds_np[i]:+0.6f}, true={targets_np[i]:+0.6f}")


Using checkpoint: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_46\checkpoints\eurusd_h1_tft_v3-epoch=45-val_loss=0.000974.ckpt


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\c

Test MAE (log-return units): 0.00099368
Direction accuracy: 0.5050 (50.50%)

Sample comparison (first 5):
0: pred=+0.000390, true=+0.000372
1: pred=+0.000231, true=+0.000483
2: pred=+0.000635, true=+0.000214
3: pred=+0.000561, true=+0.000121
4: pred=-0.000329, true=+0.000260
